# 00 - Data Audit for MBG Crawl Dataset

Notebook ini mengaudit dataset mentah di `data/raw` untuk memvalidasi struktur data, kandidat kolom teks, kandidat kolom label, kualitas data dasar, dan kesiapan dataset prototype sebelum tahap preprocessing dan labeling.

In [ ]:
from __future__ import annotations

import platform
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mbg_sentiment.data_audit import (  # noqa: E402
    audit_raw_datasets,
    file_sha256,
    infer_label_columns,
    infer_text_columns,
    label_distribution,
    list_raw_data_files,
    load_table,
    profile_dataframe,
    text_column_stats,
)

RAW_DIR = REPO_ROOT / "data" / "raw"
FIGURES_DIR = REPO_ROOT / "outputs" / "figures"
REPORTS_DIR = REPO_ROOT / "outputs" / "reports"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

## Load and Audit Raw Files

In [ ]:
raw_files = list_raw_data_files(RAW_DIR)
audit = audit_raw_datasets(RAW_DIR)

print("Raw files found:")
for raw_file in raw_files:
    print(f"- {raw_file}")

if not raw_files:
    print("No raw dataset files found in data/raw.")

In [ ]:
dataset_records = []

for file_entry in audit["datasets"]:
    path = Path(file_entry["path"])
    print("\n" + "=" * 80)
    print(f"File: {file_entry['name']}")
    print(f"SHA-256 prefix: {file_entry['sha256']}")

    for sheet_name, sheet_audit in file_entry["sheets"].items():
        print("-" * 80)
        print(f"Sheet: {sheet_name}")
        if "error" in sheet_audit:
            print(f"ERROR: {sheet_audit['error']}")
            continue

        profile = sheet_audit["profile"]
        text_cols = sheet_audit["inferred_text_columns"]
        label_cols = sheet_audit["inferred_label_columns"]

        dataset_records.append(
            {
                "file_path": path,
                "file_name": file_entry["name"],
                "sheet_name": sheet_name,
                "n_rows": profile["n_rows"],
                "n_columns": profile["n_columns"],
                "has_in_filename": "(in)" in file_entry["name"].lower(),
                "has_text_column": bool(text_cols),
                "text_columns": text_cols,
                "label_columns": label_cols,
                "sha256": file_entry["sha256"],
            }
        )

        print(f"Shape: ({profile['n_rows']}, {profile['n_columns']})")
        print(f"Columns: {profile['column_names']}")
        print("Missing values summary:")
        display(
            pd.DataFrame(
                {
                    "missing_values": profile["missing_values"],
                    "missing_rate": profile["missing_rate"],
                }
            )
            .sort_values("missing_values", ascending=False)
            .head(20)
        )
        print(f"Inferred text columns: {text_cols}")
        print(f"Inferred label columns: {label_cols}")
        print("Sample rows:")
        display(pd.DataFrame(profile["sample_rows"]))

        for text_col in text_cols:
            print(f"Text stats for column: {text_col}")
            display(pd.Series(sheet_audit["text_stats"][text_col]).to_frame("value"))

        if label_cols:
            print(f"Label distribution for column: {label_cols[0]}")
            display(pd.DataFrame(sheet_audit["label_distributions"][label_cols[0]]))

datasets_df = pd.DataFrame(dataset_records)
display(datasets_df)

## Automatic Candidate Dataset Selection

In [ ]:
if datasets_df.empty:
    selected = None
    selected_df = pd.DataFrame()
    selected_text_col = None
    selected_label_col = None
    selected_profile = {}
    selected_text_stats = {}
else:
    ranked = datasets_df.sort_values(
        by=["has_in_filename", "n_rows", "has_text_column"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    selected = ranked.iloc[0].to_dict()
    selected_df = load_table(
        selected["file_path"],
        None if selected["sheet_name"] == "<csv>" else selected["sheet_name"],
    )
    selected_text_col = (
        selected["text_columns"][0] if selected["text_columns"] else None
    )
    selected_label_col = (
        selected["label_columns"][0] if selected["label_columns"] else None
    )
    selected_profile = profile_dataframe(selected_df)
    selected_text_stats = (
        text_column_stats(selected_df, selected_text_col)
        if selected_text_col
        else {}
    )

print("Selected dataset choice:")
print(f"- file: {selected['file_name'] if selected else None}")
print(f"- sheet: {selected['sheet_name'] if selected else None}")
print(f"- text column: {selected_text_col}")
print(f"- label column: {selected_label_col}")

## Basic Figures

In [ ]:
saved_figures = []

if not datasets_df.empty:
    rows_plot_df = datasets_df.copy()
    rows_plot_df["dataset"] = rows_plot_df["file_name"] + " | " + rows_plot_df["sheet_name"].astype(str)
    plt.figure(figsize=(10, 5))
    plt.barh(rows_plot_df["dataset"], rows_plot_df["n_rows"], color="#4C78A8")
    plt.xlabel("Rows")
    plt.title("Raw Rows by File/Sheet")
    plt.tight_layout()
    path_rows = FIGURES_DIR / "00_raw_rows_by_file.png"
    plt.savefig(path_rows, dpi=150)
    plt.close()
    saved_figures.append(path_rows)

if selected is not None and not selected_df.empty:
    missing_series = selected_df.isna().mean().sort_values(ascending=False).head(20)
    plt.figure(figsize=(10, 5))
    plt.barh(missing_series.index.astype(str), missing_series.values, color="#F58518")
    plt.xlabel("Missing rate")
    plt.title("Top Missing Rate Columns")
    plt.tight_layout()
    path_missing = FIGURES_DIR / "00_missing_rate_top_columns.png"
    plt.savefig(path_missing, dpi=150)
    plt.close()
    saved_figures.append(path_missing)

if selected_text_col:
    text_lengths = selected_df[selected_text_col].dropna().astype(str).str.strip().str.len()
    text_lengths = text_lengths[text_lengths > 0]
    plt.figure(figsize=(10, 5))
    plt.hist(text_lengths, bins=30, color="#54A24B", edgecolor="white")
    plt.xlabel("Text length in characters")
    plt.ylabel("Frequency")
    plt.title("Text Length Distribution")
    plt.tight_layout()
    path_text = FIGURES_DIR / "00_text_length_distribution.png"
    plt.savefig(path_text, dpi=150)
    plt.close()
    saved_figures.append(path_text)

if selected_label_col:
    label_df = label_distribution(selected_df, selected_label_col)
    plt.figure(figsize=(8, 5))
    plt.bar(label_df["label"].astype(str), label_df["count"], color="#B279A2")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.title("Label Distribution")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    path_label = FIGURES_DIR / "00_label_distribution.png"
    plt.savefig(path_label, dpi=150)
    plt.close()
    saved_figures.append(path_label)

print("Figures saved:")
for figure in saved_figures:
    print(f"- {figure}")

## Verification Report

In [ ]:
report_path = REPORTS_DIR / "00_data_audit_verification.md"

missing_top = pd.Series(dtype=float)
duplicate_text_rate = 0.0
label_summary_markdown = "No existing label column detected"

if selected is not None and not selected_df.empty:
    missing_top = selected_df.isna().mean().sort_values(ascending=False).head(10)
    if selected_text_col and selected_text_stats:
        non_empty_text = max(
            selected_text_stats["non_null_count"]
            - selected_text_stats["empty_string_count"],
            0,
        )
        duplicate_text_rate = (
            selected_text_stats["duplicate_text_count"] / non_empty_text
            if non_empty_text > 0
            else 0.0
        )
    if selected_label_col:
        label_summary_markdown = label_distribution(
            selected_df,
            selected_label_col,
        ).to_markdown(index=False)

figure_checks = {
    "00_raw_rows_by_file.png": FIGURES_DIR / "00_raw_rows_by_file.png",
    "00_missing_rate_top_columns.png": FIGURES_DIR / "00_missing_rate_top_columns.png",
    "00_text_length_distribution.png": FIGURES_DIR / "00_text_length_distribution.png",
}
if selected_label_col:
    figure_checks["00_label_distribution.png"] = FIGURES_DIR / "00_label_distribution.png"

checks = {
    "at least one raw dataset found": len(raw_files) > 0,
    "selected dataset has > 0 rows": bool(selected_profile.get("n_rows", 0) > 0),
    "selected dataset has at least one inferred text column": bool(selected_text_col),
    "selected text column has non-null text": bool(
        selected_text_stats.get("non_null_count", 0) > 0
    ),
    "duplicate text rate is reported": duplicate_text_rate >= 0,
    "figures were saved": all(path.exists() for path in figure_checks.values()),
    "report file was saved": True,
}

key_findings = []
if raw_files:
    key_findings.append(f"Found {len(raw_files)} raw dataset file(s) in data/raw.")
else:
    key_findings.append("No raw dataset files were found in data/raw.")
if selected:
    key_findings.append(
        f"Selected dataset is {selected['file_name']} sheet {selected['sheet_name']} "
        f"with {selected_profile.get('n_rows', 0)} rows and "
        f"{selected_profile.get('n_columns', 0)} columns."
    )
if selected_text_col:
    key_findings.append(
        f"Selected text column '{selected_text_col}' has "
        f"{selected_text_stats.get('non_null_count', 0)} non-null values, "
        f"{selected_text_stats.get('empty_string_count', 0)} empty strings, and "
        f"{selected_text_stats.get('duplicate_text_count', 0)} duplicate text values."
    )
else:
    key_findings.append("No likely text/comment column was detected.")
if selected_label_col:
    key_findings.append(f"Detected label column candidate: {selected_label_col}.")
else:
    key_findings.append("No existing label column was detected.")

report_lines = [
    "# 00 Data Audit Verification",
    "",
    "## Environment",
    f"- Python version: {platform.python_version()}",
    f"- pandas version: {pd.__version__}",
    f"- numpy version: {np.__version__}",
    f"- matplotlib version: {matplotlib.__version__}",
    f"- seed: {SEED}",
    "",
    "## Repository/data status",
    "- raw files found:",
]
report_lines.extend([f"  - {path.name}" for path in raw_files] or ["  - none"])
report_lines.extend(
    [
        f"- selected dataset: {selected['file_name'] if selected else 'None'}",
        f"- selected sheet: {selected['sheet_name'] if selected else 'None'}",
        f"- selected text column: {selected_text_col}",
        f"- selected label column: {selected_label_col if selected_label_col else 'None'}",
        "",
        "## Dataset fingerprint",
        f"- n_rows: {selected_profile.get('n_rows', 0)}",
        f"- n_features: {selected_profile.get('n_columns', 0)}",
        f"- sha256 first 12 chars: {selected['sha256'] if selected else 'None'}",
        "",
        "## Data quality",
        "- missing values top columns:",
    ]
)
if not missing_top.empty:
    report_lines.extend(
        [f"  - {col}: {rate:.4f}" for col, rate in missing_top.items()]
    )
else:
    report_lines.append("  - none")
report_lines.extend(
    [
        f"- duplicate row count: {selected_profile.get('duplicate_rows', 0)}",
        f"- duplicate text count: {selected_text_stats.get('duplicate_text_count', 0)}",
        f"- duplicate text rate: {duplicate_text_rate:.4f}",
        f"- empty text count: {selected_text_stats.get('empty_string_count', 0)}",
        f"- average text length: {selected_text_stats.get('avg_length_chars', 0):.2f}",
        f"- median text length: {selected_text_stats.get('median_length_chars', 0):.2f}",
        f"- average word count: {selected_text_stats.get('avg_word_count', 0):.2f}",
        f"- median word count: {selected_text_stats.get('median_word_count', 0):.2f}",
        "",
        "## Label summary",
        label_summary_markdown,
        "",
        "## Key findings",
    ]
)
report_lines.extend([f"- {finding}" for finding in key_findings])
report_lines.extend(
    [
        "",
        "## Self-run acceptance checks",
    ]
)
report_lines.extend(
    [f"- {'PASS' if status else 'FAIL'}: {name}" for name, status in checks.items()]
)
report_lines.extend(
    [
        "",
        "## Figures generated",
    ]
)
report_lines.extend(
    [f"- {path}" for path in saved_figures] or ["- none"]
)
report_lines.extend(
    [
        "",
        "## Next recommended step",
        "Prepare a cleaned interim dataset, then continue with labeling validation.",
    ]
)

report_path.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Verification report saved to: {report_path}")